# Fifteen-class design: eligibility, sampling, and validation folds

1. Define the 15 classes: ten focal games $F_1,\dots,F_{10}$ and five background pools $G_1,\dots,G_5$
   holding the other 90 games.
2. Apply the eligibility rules to every review of the 100 games and audit the counts.
3. Sample 100,000 reviews uniformly from each focal game.
4. Sample 100,000 reviews from each pool under a common year distribution, spread as evenly as
   availability permits across the pool's games within each year.
5. Assign validation folds: by review (stratified by year) for focal games, by whole game for pools.
6. Save the sample with text, the embedding row index for focal reviews, and the audit tables.

Embedding and classifier fitting happen on the cluster from the saved sample.
Everything written here goes to `results/design15/`.

## 0. Setup and class definitions

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import gc, re, time, zipfile
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
DATA = ROOT / "data"
NEW = ROOT / "game data 2"
OUT = ROOT / "results" / "design15"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
N_FOLDS = 5
N_PER_CLASS = 100_000
YEARS = list(range(2015, 2025))
WORD_LIMIT = 350          # reviews longer than this exceed the encoder's token window (see slides, section 1)

# the ten focal files in the order the embedding matrix was built:
# row i of data/embeddings_v2.npz is row i of this concatenation
FOCAL_FILES = [
    "game_reviews_218620.feather",  # PAYDAY 2
    "game_reviews_236390.feather",  # War Thunder
    "game_reviews_251570.feather",  # 7 Days to Die
    "game_reviews_284160.feather",  # BeamNG.drive
    "game_reviews_304930.feather",  # Unturned
    "game_reviews_377160.feather",  # Fallout 4
    "game_reviews_381210.feather",  # Dead by Daylight
    "game_reviews_413150.feather",  # Stardew Valley
    "game_reviews_438100.feather",  # VRChat
    "game_reviews_444090.feather",  # Paladins
]
FOCAL = {
    251570: ("F1", "7 Days to Die"),
    284160: ("F2", "BeamNG.drive"),
    381210: ("F3", "Dead by Daylight"),
    377160: ("F4", "Fallout 4"),
    218620: ("F5", "PAYDAY 2"),
    444090: ("F6", "Paladins"),
    413150: ("F7", "Stardew Valley"),
    304930: ("F8", "Unturned"),
    438100: ("F9", "VRChat"),
    236390: ("F10", "War Thunder"),
}
GROUP_LABELS = {
    "G1": "Shooters and combat/action",
    "G2": "Role-playing and narrative/action adventure",
    "G3": "Survival, crafting, and exploration",
    "G4": "Simulation, creative play, and physics/puzzles",
    "G5": "Strategy, management, and tactical planning",
}
GROUPS = {
    "G1": {
        1085660: "Destiny 2",
        632360: "Risk of Rain 2",
        311210: "Call of Duty®: Black Ops III",
        367520: "Hollow Knight",
        548430: "Deep Rock Galactic",
        107410: "Arma 3",
        594650: "Hunt: Showdown",
        393380: "Squad",
        379720: "DOOM",
        677620: "Splitgate",
        700330: "SCP: Secret Laboratory",
        220: "Half-Life 2",
        304390: "FOR HONOR™",
        629760: "MORDHAU",
        386360: "SMITE®",
        581320: "Insurgency: Sandstorm",
        70: "Half-Life",
        433850: "Z1 Battle Royale",
        552500: "Warhammer: Vermintide 2",
        227940: "Heroes & Generals",
        301520: "Robocraft",
        674940: "Stick Fight: The Game",
        552520: "Far Cry® 5",
        814380: "Sekiro™: Shadows Die Twice - GOTY Edition",
        232090: "Killing Floor 2",
        268910: "Cuphead",
        240: "Counter-Strike: Source",
        8870: "BioShock Infinite",
        552990: "World of Warships",
        10: "Counter-Strike",
        222880: "Insurgency",
        588650: "Dead Cells",
        362890: "Black Mesa",
        601150: "Devil May Cry 5",
        976310: "Mortal Kombat 11",
        466240: "Deceit",
        774171: "Muse Dash",
        755790: "Ring of Elysium",
        365590: "Tom Clancy’s The Division™",
        444200: "World of Tanks Blitz",
    },
    "G2": {
        1174180: "Red Dead Redemption 2",
        489830: "The Elder Scrolls V: Skyrim Special Edition",
        582010: "Monster Hunter: World",
        374320: "DARK SOULS™ III",
        391540: "Undertale",
        698780: "Doki Doki Literature Club!",
        238960: "Path of Exile",
        435150: "Divinity: Original Sin 2 - Definitive Edition",
        306130: "The Elder Scrolls® Online",
        319630: "Life is Strange - Episode 1",
        203160: "Tomb Raider",
        12210: "Grand Theft Auto IV: The Complete Edition",
        225540: "Just Cause™ 3",
        812140: "Assassin's Creed® Odyssey",
        524220: "NieR:Automata™",
        287700: "METAL GEAR SOLID V: THE PHANTOM PAIN",
        379430: "Kingdom Come: Deliverance",
        208650: "Batman™: Arkham Knight",
        304050: "Trove",
        356190: "Middle-earth™: Shadow of War™",
        219990: "Grim Dawn",
        883710: "Resident Evil 2",
        238320: "Outlast",
        582160: "Assassin's Creed® Origins",
        391220: "Rise of the Tomb Raider™",
    },
    "G3": {
        275850: "No Man's Sky",
        221100: "DayZ",
        108600: "Project Zomboid",
        361420: "ASTRONEER",
        387990: "Scrap Mechanic",
        305620: "The Long Dark",
    },
    "G4": {
        1118200: "People Playground",
        270880: "American Truck Simulator",
        400: "Portal",
        220200: "Kerbal Space Program",
        433340: "Slime Rancher",
        518790: "theHunter: Call of the Wild™",
        477160: "Human: Fall Flat",
        244210: "Assetto Corsa",
    },
    "G5": {
        394360: "Hearts of Iron IV",
        294100: "RimWorld",
        289070: "Sid Meier’s Civilization® VI",
        427520: "Factorio",
        508440: "Totally Accurate Battle Simulator",
        236850: "Europa Universalis IV",
        646570: "Slay the Spire",
        221380: "Age of Empires II (2013)",
        813780: "Age of Empires II: Definitive Edition",
        231430: "Company of Heroes 2",
        457140: "Oxygen Not Included",
    },
}

CLASSES = [f"F{j}" for j in range(1, 11)] + [f"G{m}" for m in range(1, 6)]
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}

rows = []
for f in FOCAL_FILES:
    appid = int(f.split("_")[-1].split(".")[0])
    cls, name = FOCAL[appid]
    rows.append(dict(appid=appid, name=name, cls=cls, path=ROOT / "data" / "raw" / f))
for g, members in GROUPS.items():
    for appid, name in members.items():
        rows.append(dict(appid=appid, name=name, cls=g, path=NEW / "90 popular games" / f"game_reviews_{appid}.feather"))
games = pd.DataFrame(rows)
games["class_idx"] = games["cls"].map(CLASS_IDX)
NAME2 = dict(zip(games["appid"], games["name"]))

assert len(games) == 100 and games["appid"].is_unique
assert all(p.exists() for p in games["path"]), [p for p in games["path"] if not p.exists()]
assert set(games.loc[games.cls.str.startswith("G"), "appid"]) == {int(p.stem.split("_")[-1]) for p in (NEW / "90 popular games").glob("*.feather")}
assert not set(FOCAL) & {a for m in GROUPS.values() for a in m}
print(games.groupby("cls", sort=False).size().to_string())


cls
F5      1
F10     1
F1      1
F2      1
F8      1
F4      1
F3      1
F7      1
F9      1
F6      1
G1     40
G2     25
G3      6
G4      8
G5     11


## 1. Eligibility

A review is eligible when it is tagged English, has at least one word, was created in 2015–2024,
and does not name any of the 100 games under the title-matching rule below (the same patterns as
section 12 of `information_in_reviews.ipynb`). Repeated records of the same review are removed by
`recommendationid` before any other filter. The audit table records how many reviews each rule removes
from each game. For focal games the position of every review in the concatenated embedding matrix is
kept (`emb_row`), so the cluster can take their vectors from `embeddings_v2.npz` without re-encoding.
The text itself is not retained in this pass; it is read again only for the sampled reviews.
The pass takes about 25 minutes over the 20.7 million raw records, so its result is cached in
`results/design15/eligible_meta.parquet`; delete that file to force a recount.

In [2]:
# name, franchise name or common abbreviation of each game; matched on lower-cased text as in section 1
LOWER = {
    218620: r"pay\s?day", 236390: r"war\s?thunder", 251570: r"7\s?days|seven days|7dtd|7d2d",
    284160: r"beam\s?ng", 304930: r"unturned", 377160: r"fallout|\bfo4\b",
    381210: r"dead by daylight|\bdbd\b", 413150: r"stardew", 438100: r"vr\s?chat", 444090: r"\bpaladins\b",
    10: r"counter[\s-]?strike|\bcs ?1\.6\b|\bcsgo\b|\bcs:go\b|\bcss\b|\bcs\b", 240: r"counter[\s-]?strike|\bcs ?1\.6\b|\bcsgo\b|\bcs:go\b|\bcss\b|\bcs\b",
    70: r"half[\s-]?life|\bhl ?[12]\b|\bhl\b", 220: r"half[\s-]?life|\bhl ?[12]\b|\bhl\b",
    8870: r"bio\s?shock", 12210: r"grand theft auto|\bgta\b", 107410: r"\barma ?[23]?\b", 108600: r"zomboid",
    203160: r"tomb raider|lara croft", 391220: r"tomb raider|lara croft", 208650: r"arkham|\bbatman\b",
    219990: r"grim dawn", 220200: r"kerbal|\bksp\b", 221100: r"\bday ?z\b",
    221380: r"age of empires|\baoe ?\d?\b", 813780: r"age of empires|\baoe ?\d?\b",
    222880: r"insurgency", 581320: r"insurgency|sandstorm", 227940: r"heroes (&|and) generals|\bh&g\b",
    231430: r"company of heroes|\bcoh ?[123]?\b", 232090: r"killing floor|\bkf ?[12]\b",
    236850: r"europa universalis|\beu ?(4|iv)\b", 238960: r"path of exile|\bpoe\b", 244210: r"assetto",
    268910: r"cuphead", 270880: r"american truck|\bats\b", 275850: r"no man.?s sky|\bnms\b",
    287700: r"metal gear|\bmgs ?(v|5)?\b|phantom pain",
    289070: r"sid meier|civilization ?(vi|6|v|5|iv|4)\b|\bciv ?(vi|6|v|5|iv|4)?\b",
    294100: r"rim\s?world", 301520: r"robocraft", 305620: r"the long dark|\btld\b",
    306130: r"elder scrolls online|\beso\b", 311210: r"call of duty|\bcod\b|black ops|\bbo ?[1234]\b",
    319630: r"life is strange|\blis\b", 356190: r"shadow of (war|mordor)|middle[\s-]?earth",
    361420: r"astroneer", 362890: r"black mesa", 365590: r"\bthe division\b|tom clancy", 367520: r"hollow knight",
    374320: r"dark souls|\bds ?[123]\b", 379430: r"kingdom come|\bkcd\b", 387990: r"scrap mechanic",
    391540: r"undertale", 394360: r"hearts of iron|\bhoi ?(4|iv)?\b", 427520: r"factorio",
    433340: r"slime rancher", 433850: r"z1 battle royale|\bh1z1\b|\bz1\b", 435150: r"divinity|\bdos ?2\b",
    444200: r"world of tanks|\bwot\b", 457140: r"oxygen not included|\boni\b", 477160: r"human:? fall flat|\bhff\b",
    489830: r"skyrim|elder scrolls v\b|\btes ?(v|5)\b", 508440: r"totally accurate battle|\btabs\b",
    518790: r"the ?hunter|call of the wild|\bcotw\b", 524220: r"nier|automata", 548430: r"deep rock|\bdrg\b",
    552500: r"vermintide|warhammer", 552520: r"far ?cry", 552990: r"world of warships|\bwows\b",
    582010: r"monster hunter|\bmhw\b", 582160: r"assassin.?s creed|\bac ?(origins|odyssey)\b",
    812140: r"assassin.?s creed|\bac ?(origins|odyssey)\b", 588650: r"dead cells",
    594650: r"hunt:? showdown|huntshowdown", 601150: r"devil may cry|\bdmc ?5?\b", 629760: r"mordhau",
    632360: r"risk of rain|\bror ?2?\b", 646570: r"slay the spire|\bsts\b", 674940: r"stick fight",
    677620: r"splitgate", 698780: r"doki doki|\bddlc\b", 700330: r"scp:? secret lab|\bscp ?sl\b|\bscp\b",
    755790: r"ring of elysium|\broe\b", 774171: r"muse dash", 814380: r"sekiro",
    883710: r"resident evil|\bre2\b", 976310: r"mortal kombat|\bmk ?11\b", 1118200: r"people playground",
    1174180: r"red dead|\brdr ?2?\b",
}
# names that are ordinary English words: matched case-sensitively, capitalised as a title
EXACT = {
    400: r"\bPortal ?[12]?\b", 225540: r"\bJust Cause ?[1234]?\b", 238320: r"\bOutlast ?[12]?\b",
    304050: r"\bTrove\b", 304390: r"\bFor Honou?r\b", 379720: r"\bDOOM\b|\bDoom\b", 386360: r"\bSMITE\b|\bSmite\b",
    393380: r"\bSquad\b", 466240: r"\bDeceit\b", 1085660: r"\bDestiny ?2?\b|destiny ?2\b",
}
assert set(LOWER) | set(EXACT) == set(NAME2), set(NAME2) ^ (set(LOWER) | set(EXACT))
LOWER_RE = "|".join(re.sub(r"\((?!\?)", "(?:", p) for p in sorted(set(LOWER.values())))
EXACT_RE = "|".join(re.sub(r"\((?!\?)", "(?:", p) for p in sorted(set(EXACT.values())))


In [3]:
with zipfile.ZipFile(DATA / "embeddings_v2.npz") as z:
    with z.open("X.npy") as f:
        version = np.lib.format.read_magic(f)
        emb_shape, _, _ = np.lib.format._read_array_header(f, version)

COLS = ["recommendationid", "language", "review", "timestamp_created", "timestamp_updated",
        "voted_up", "votes_up", "votes_funny", "weighted_vote_score"]
CACHE = OUT / "eligible_meta.parquet"
if CACHE.exists():
    elig = pd.read_parquet(CACHE)
    audit = pd.read_csv(OUT / "eligibility_audit.csv")
    offset = emb_shape[0]
    print(f"loaded cached eligibility pass: {len(elig):,} eligible reviews")
else:
    meta_parts, audit_rows = [], []
    offset, t0 = 0, time.time()
    for g in games.itertuples():
        d = pd.read_feather(g.path, columns=COLS)
        d["src_pos"] = np.arange(len(d))
        if g.cls.startswith("F"):
            d["emb_row"] = np.arange(offset, offset + len(d))
            offset += len(d)
        else:
            d["emb_row"] = -1
        n_total = len(d)
        d = d[~d["recommendationid"].duplicated(keep="first")]
        n_dedup = len(d)
        d = d[d["language"] == "english"]
        n_eng = len(d)
        text = d["review"].fillna("").astype(str)
        wc = text.str.count(r"\S+").astype(int)
        keep = (wc >= 1).to_numpy()
        d, text, wc = d[keep], text[keep], wc[keep]
        n_word = len(d)
        year = pd.to_datetime(d["timestamp_created"], unit="s").dt.year
        keep = year.between(YEARS[0], YEARS[-1]).to_numpy()
        d, text, wc, year = d[keep], text[keep], wc[keep], year[keep]
        n_year = len(d)
        named = (text.str.lower().str.contains(LOWER_RE, regex=True)
                 | text.str.contains(EXACT_RE, regex=True)).to_numpy()
        d, wc, year = d[~named], wc[~named], year[~named]
        n_elig = len(d)
        meta_parts.append(pd.DataFrame({
            "appid": g.appid, "cls": g.cls,
            "recommendationid": d["recommendationid"].astype(str).to_numpy(),
            "year": year.to_numpy().astype(np.int16),
            "word_count": wc.to_numpy().astype(np.int32),
            "emb_row": d["emb_row"].to_numpy().astype(np.int64),
            "src_pos": d["src_pos"].to_numpy().astype(np.int64),
            "edited": (d["timestamp_updated"] != d["timestamp_created"]).to_numpy(),
            "voted_up": d["voted_up"].fillna(False).astype(bool).to_numpy(),
            "votes_up": pd.to_numeric(d["votes_up"]).fillna(0).to_numpy().astype(np.int32),
            "votes_funny": pd.to_numeric(d["votes_funny"]).fillna(0).to_numpy().astype(np.int32),
            "weighted_vote_score": pd.to_numeric(d["weighted_vote_score"]).fillna(0).to_numpy().astype(np.float32),
        }))
        audit_rows.append(dict(appid=g.appid, name=g.name, cls=g.cls, total=n_total, after_dedup=n_dedup,
                               english=n_eng, at_least_one_word=n_word, in_window=n_year, eligible=n_elig))
        del d, text, wc, year, named
        gc.collect()
        print(f"{g.cls:>3} {g.name[:42]:<42} {n_total:>9,} -> {n_elig:>8,} eligible   ({time.time() - t0:5.0f}s)")

    assert offset == emb_shape[0], (offset, emb_shape)
    elig = pd.concat(meta_parts, ignore_index=True)
    del meta_parts
    gc.collect()
    audit = pd.DataFrame(audit_rows)
    elig.to_parquet(CACHE, index=False)
audit.to_csv(OUT / "eligibility_audit.csv", index=False)
print(f"\n{len(elig):,} eligible reviews across 100 games; focal rows aligned with the {emb_shape} embedding matrix")


 F5 PAYDAY 2                                     657,871 ->  207,466 eligible   (   15s)
F10 War Thunder                                  558,080 ->  211,165 eligible   (   33s)
 F1 7 Days to Die                                233,300 ->  103,033 eligible   (   45s)
 F2 BeamNG.drive                                 281,539 ->  151,542 eligible   (   51s)
 F8 Unturned                                     443,297 ->  155,600 eligible   (   59s)
 F4 Fallout 4                                    380,874 ->  170,371 eligible   (   78s)
 F3 Dead by Daylight                             755,139 ->  282,071 eligible   (  105s)
 F7 Stardew Valley                               780,466 ->  354,979 eligible   (  129s)
 F9 VRChat                                       228,119 ->  140,359 eligible   (  138s)
 F6 Paladins                                     304,797 ->  102,164 eligible   (  145s)
 G1 Destiny 2                                    615,538 ->  326,025 eligible   (  177s)
 G1 Risk of Rain 2   

In [4]:
# what each rule removes, summed over the 100 games
stages = ["total", "after_dedup", "english", "at_least_one_word", "in_window", "eligible"]
tot = audit[stages].sum()
summary = pd.DataFrame({"reviews": tot, "removed_by_this_rule": tot.shift(1) - tot}).fillna(0).astype(int)
summary


,reviews,removed_by_this_rule
total,20702579,0
after_dedup,20702578,1
english,9397337,11305241
at_least_one_word,9365195,32142
in_window,9114490,250705
eligible,8092853,1021637


### 1a. Eligible inventory by game and year

In [5]:
inv = (elig.pivot_table(index="appid", columns="year", values="src_pos", aggfunc="size")
           .reindex(columns=YEARS, fill_value=0).fillna(0).astype(int))
inv.insert(0, "cls", inv.index.map(dict(zip(games.appid, games.cls))))
inv.insert(1, "name", inv.index.map(NAME2))
inv["total"] = inv[YEARS].sum(1)
inv = inv.sort_values(["cls", "total"], ascending=[True, False], key=lambda s: s.map(CLASS_IDX) if s.name == "cls" else s)
inv.to_csv(OUT / "eligible_game_year.csv")

# the previous inventory (section 12 of the main notebook) counted the same rules without deduplication
prev = pd.read_csv(ROOT / "results" / "eligible_reviews_per_game_year.csv").set_index("appid")
diff = (inv[YEARS] - prev.loc[inv.index, [str(y) for y in YEARS]].to_numpy()).abs().sum().sum()
print(f"absolute difference from the previous inventory over 2015–2024: {diff:,} reviews")

class_year = inv.groupby("cls")[YEARS].sum().reindex(CLASSES)
class_year["total"] = class_year.sum(1)
class_year["games"] = games.groupby("cls").size().reindex(CLASSES)
class_year.to_csv(OUT / "eligible_class_year.csv")
pd.set_option("display.width", 250, "display.max_rows", 120)
class_year


absolute difference from the previous inventory over 2015–2024: 0 reviews


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,total,games
cls,,,,,,,,,,,,
F1,2897,3910,5304,5756,14033,24022,22502,19496,5113,0,103033,1
F2,1221,1970,2307,2197,7201,16590,26122,27510,33535,32889,151542,1
F3,0,16169,13791,14022,28684,51685,63453,36257,31139,26871,282071,1
F4,14772,14579,16490,7473,12272,21962,19703,16960,15336,30824,170371,1
F5,30654,18846,25707,9891,15132,25880,31763,22817,19206,7570,207466,1
F6,0,4260,22579,11290,13471,19305,13106,8765,5535,3853,102164,1
F7,0,23892,15227,13769,39878,50851,59166,48634,47820,55742,354979,1
F8,9840,22560,18052,8114,14614,26392,20169,15114,12824,7921,155600,1
F9,0,0,1142,8833,8290,20183,25809,43696,16641,15765,140359,1


Every focal game has more than 100,000 eligible reviews in 2015–2024, so the focal classes need no
exception. For the pools the binding year
is 2015, where $G_5$ has the fewest eligible reviews.

In [6]:
focal_tot = class_year.loc[[c for c in CLASSES if c.startswith("F")], "total"]
assert (focal_tot >= N_PER_CLASS).all(), focal_tot[focal_tot < N_PER_CLASS]
pool_year = class_year.loc[[c for c in CLASSES if c.startswith("G")], YEARS]
common_capacity = pool_year.min(axis=0)
print("smallest focal count:", f"{focal_tot.min():,}", f"({focal_tot.idxmin()})")
print("common capacity by year (min over pools):")
print(common_capacity.to_string())


smallest focal count: 102,164 (F6)
common capacity by year (min over pools):
year
2015     9115
2016    18654
2017    20811
2018    19238
2019    37408
2020    71504
2021    65846
2022    86910
2023    74792
2024    53520


## 2. Focal sample

100,000 distinct eligible reviews are drawn uniformly without replacement from each focal game's full
2015–2024 history. There are no year quotas, so each game's own temporal composition is preserved in
expectation.

In [7]:
rng = np.random.default_rng(SEED)
focal_pick = []
for appid in FOCAL:
    idx = elig.index[elig["appid"] == appid].to_numpy()
    focal_pick.append(rng.choice(idx, size=N_PER_CLASS, replace=False))
focal_pick = np.concatenate(focal_pick)
focal_sample = elig.loc[focal_pick].copy()
assert focal_sample.groupby("cls").size().eq(N_PER_CLASS).all()
focal_year = focal_sample.pivot_table(index="cls", columns="year", values="src_pos", aggfunc="size").reindex(CLASSES[:10]).fillna(0).astype(int)
focal_year


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
cls,,,,,,,,,,
F1,2821,3786,5142,5576,13638,23336,21777,18953,4971,0
F2,800,1342,1555,1425,4766,10934,17323,18173,22118,21564
F3,0,5786,4922,4951,10070,18161,22645,12863,11190,9412
F4,8743,8473,9707,4412,7191,12960,11378,10000,8977,18159
F5,14726,9165,12434,4763,7292,12394,15318,10927,9270,3711
F6,0,4173,22063,11043,13197,18909,12817,8586,5426,3786
F7,0,6735,4251,3862,11272,14374,16603,13746,13353,15804
F8,6346,14472,11700,5222,9393,16934,12999,9687,8230,5017
F9,0,0,837,6248,5950,14353,18325,31163,11816,11308


## 3. Pool sample under a common year distribution

**Year quotas.** Let $N_{my}$ be the eligible count of pool $m$ in year $y$ and $C_y=\min_m N_{my}$.
The common quota $q_y$ is the capped equal allocation: the largest integer level $\lambda$ with
$\sum_y \min(C_y,\lambda)\le 100{,}000$, and the remaining units go one each to unsaturated years in
chronological order. This gives the quotas stated in the design: 9,115 in 2015, 10,099 in 2016–2018,
10,098 in 2019–2024.

**Within a pool-year.** The same rule allocates $q_y$ across the pool's games with capacities $N_{amy}$;
remaining units go to unsaturated games in ascending order of app id. A game with fewer eligible reviews
than its equal share contributes all of them, and the rest is spread over the other games. Reviews are
then drawn uniformly without replacement inside each game-year cell.

In [8]:
def capped_equal_allocation(capacity, total):
    """Largest common integer level whose capped allocations do not exceed `total`;
    the remaining units go one each to unsaturated entries in index order."""
    cap = pd.Series(capacity).astype(int)
    assert cap.sum() >= total, (cap.sum(), total)
    lo, hi = 0, int(cap.max())
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if np.minimum(cap, mid).sum() <= total:
            lo = mid
        else:
            hi = mid - 1
    alloc = np.minimum(cap, lo).copy()
    rem = total - alloc.sum()
    for i in cap.index:
        if rem == 0:
            break
        if alloc[i] < cap[i]:
            alloc[i] += 1
            rem -= 1
    assert rem == 0 and (alloc <= cap).all() and alloc.sum() == total
    return alloc

quota = capped_equal_allocation(common_capacity, N_PER_CLASS)
expected = pd.Series({2015: 9115, **{y: 10099 for y in (2016, 2017, 2018)}, **{y: 10098 for y in range(2019, 2025)}})
assert quota.equals(expected.reindex(YEARS)), quota
quotas = pd.DataFrame({"common_capacity": common_capacity, "quota_per_pool": quota, "all_five_pools": 5 * quota})
quotas.loc["total"] = quotas.sum()
quotas.to_csv(OUT / "year_quotas.csv")
quotas


,common_capacity,quota_per_pool,all_five_pools
year,,,
2015,9115,9115,45575
2016,18654,10099,50495
2017,20811,10099,50495
2018,19238,10099,50495
2019,37408,10098,50490
2020,71504,10098,50490
2021,65846,10098,50490
2022,86910,10098,50490
2023,74792,10098,50490


In [9]:
alloc_rows, pool_pick = [], []
for m in [c for c in CLASSES if c.startswith("G")]:
    members = sorted(GROUPS[m])
    sub = elig[elig["cls"] == m]
    cells = sub.groupby(["appid", "year"]).indices
    for y in YEARS:
        cap = pd.Series({a: len(cells.get((a, y), ())) for a in members})
        alloc = capped_equal_allocation(cap, int(quota[y]))
        for a in members:
            n = int(alloc[a])
            alloc_rows.append(dict(cls=m, appid=a, name=NAME2[a], year=y, eligible=int(cap[a]), allocated=n))
            if n:
                pool_pick.append(sub.index.to_numpy()[rng.choice(cells[(a, y)], size=n, replace=False)])
pool_pick = np.concatenate(pool_pick)
pool_sample = elig.loc[pool_pick].copy()
assert pool_sample.groupby("cls").size().eq(N_PER_CLASS).all()
assert not pool_sample["recommendationid"].duplicated().any()

allocation = pd.DataFrame(alloc_rows)
alloc_wide = allocation.pivot_table(index=["cls", "appid", "name"], columns="year", values="allocated").astype(int)
alloc_wide["total"] = alloc_wide.sum(1)
alloc_wide.to_csv(OUT / "allocation_game_year.csv")
print("games whose full availability is taken in at least one year:",
      f"{(allocation.eligible == allocation.allocated).groupby(allocation.appid).any().sum()} of 90")
alloc_wide


games whose full availability is taken in at least one year: 64 of 90


year                                                       2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  total
cls appid   name                                                                                                            
G1  10      Counter-Strike                                  652   572   399   316   253   253   253   253   253   312   3516
    70      Half-Life                                       651   572   399   316   253   253   253   253   253   312   3515
    220     Half-Life 2                                     651   572   399   316   253   253   253   253   253   312   3515
    240     Counter-Strike: Source                          651   572   399   316   253   253   253   253   253   312   3515
    8870    BioShock Infinite                               651   571   399   316   253   253   253   253   253   312   3514
    107410  Arma 3                                          651   571   399   316   253   253   253   253   253   312   3514
    222880  Insurgency                                      651   571   399   316   253   253   253   253   253   312   3514
    227940  Heroes & Generals                               651   571   398   316   253   253   253   253   253   312   3513
    232090  Killing Floor 2                                 651   571   398   316   253   253   253   253   253   312   3513
    268910  Cuphead                                           0     0   398   316   253   253   253   253   253     0   1979
    301520  Robocraft                                       651   571   398   316   253   253   253   253   253   312   3513
    304390  FOR HONOR™                                        0     0   398   316   253   253   253   253   253   312   2291
    311210  Call of Duty®: Black Ops III                    651   571   398   316   253   253   253   253   253   311   3512
    362890  Black Mesa                                      651   571   398   316   253   253   253   253   253   311   3512
    365590  Tom Clancy’s The Division™                        0   571   398   316   253   253   253   253   253   311   2861
    367520  Hollow Knight                                     0     0   398   316   253   253   253   253   253   311   2290
    379720  DOOM                                              0   571   398   316   253   253   253   253   253   311   2861
    386360  SMITE®                                          651   571   398   316   253   253   253   253   253   311   3512
    393380  Squad                                           651   571   398   316   252   252   252   252   252   311   3507
    433850  Z1 Battle Royale                                  0   571   398   315   252   252   252   252   252   247   2791
    444200  World of Tanks Blitz                              0   315   398   315   252   252   252   252   252     0   2288
    466240  Deceit                                            0    73   398   315   252   252   252   252   252   311   2357
    548430  Deep Rock Galactic                                0     0     0   315   252   252   252   252   252     0   1575
    552500  Warhammer: Vermintide 2                           0     0     0   315   252   252   252   252   252   311   1886
    552520  Far Cry® 5                                        0     0     0   315   252   252   252   252   252   311   1886
    552990  World of Warships                                 0     0   398   315   252   252   252   252   252   311   2284
    581320  Insurgency: Sandstorm                             0     0     0   315   252   252   252   252   252   311   1886
    588650  Dead Cells                                        0     0   398   315   252   252   252   252   252   311   2284
    594650  Hunt: Showdown                                    0     0     0   315   252   252   252   252   252   311   1886
    601150  Devil May Cry 5                                   0     0     0     0   252   252   252   252   252   311   1571


Planned allocation equals realised allocation by construction, since every cell is sampled exactly
at its allocation. The per-game totals below show how unequal the games' contributions are inside a pool:
early years are carried by the few games that existed then.

In [10]:
per_game = alloc_wide.reset_index().groupby("cls")["total"].agg(games="size", min="min", median="median", max="max")
per_game


,games,min,median,max
cls,,,,
G1,40,1260,2289.0,3516
G2,25,2015,4120.0,4951
G3,6,14965,15615.0,19657
G4,8,8329,12492.5,16735
G5,11,4590,9941.0,13340


## 4. Validation folds

**Focal games** are split by review: inside every game-year cell the reviews are shuffled and dealt
cyclically to the five folds, so each fold holds about a fifth of each game with its year proportions
preserved.

**Pool games** are split by game: every sampled review of a game goes to the same fold. Within a pool the
games are partitioned so that fold sizes in games differ by at most one, and among such partitions the
one is chosen that best balances review counts and year coverage across folds (random restarts followed
by pairwise swaps, minimising the coefficient of variation of fold totals plus the average coefficient of
variation of fold counts within each year). A game is never divided between folds.

In [11]:
sample = pd.concat([focal_sample, pool_sample], ignore_index=True)
sample["fold"] = -1

# focal: by review, stratified by year within game
focal_idx = sample.index[sample["cls"].str.startswith("F")]
for (appid, year), pos in sample.loc[focal_idx].groupby(["appid", "year"]).indices.items():
    idx = focal_idx[pos]
    perm = rng.permutation(len(idx))
    folds = (np.arange(len(idx)) + rng.integers(N_FOLDS)) % N_FOLDS
    sample.loc[idx[perm], "fold"] = folds

def fold_objective(assign, counts):
    """assign: fold of each game (array); counts: games x years review counts."""
    tot = np.array([counts[assign == k].sum() for k in range(N_FOLDS)], dtype=float)
    by_year = np.stack([counts[assign == k].sum(0) for k in range(N_FOLDS)])     # folds x years
    cv_tot = tot.std() / tot.mean()
    col_mean = by_year.mean(0)
    cv_year = (by_year.std(0)[col_mean > 0] / col_mean[col_mean > 0]).mean()
    return cv_tot + cv_year

def assign_games_to_folds(counts, rng, n_restarts=2000):
    n = len(counts)
    sizes = [n // N_FOLDS + (1 if k < n % N_FOLDS else 0) for k in range(N_FOLDS)]
    base = np.repeat(np.arange(N_FOLDS), sizes)
    best, best_val = None, np.inf
    for _ in range(n_restarts):
        a = base[rng.permutation(n)]
        v = fold_objective(a, counts)
        if v < best_val:
            best, best_val = a, v
    improved = True
    while improved:
        improved = False
        for i in range(n):
            for j in range(i + 1, n):
                if best[i] == best[j]:
                    continue
                a = best.copy(); a[i], a[j] = a[j], a[i]
                v = fold_objective(a, counts)
                if v < best_val - 1e-12:
                    best, best_val, improved = a, v, True
    return best, best_val

game_fold_rows = []
for m in [c for c in CLASSES if c.startswith("G")]:
    gy = alloc_wide.loc[m][YEARS]
    appids = gy.index.get_level_values("appid").to_numpy()
    counts = gy.to_numpy()
    assign, val = assign_games_to_folds(counts, rng)
    for a, k, r in zip(appids, assign, counts.sum(1)):
        game_fold_rows.append(dict(cls=m, appid=int(a), name=NAME2[a], fold=int(k), reviews=int(r)))
    print(f"{m}: objective {val:.3f}")
game_folds = pd.DataFrame(game_fold_rows)
game_folds.to_csv(OUT / "game_folds.csv", index=False)

fold_of_game = dict(zip(game_folds["appid"], game_folds["fold"]))
is_pool = sample["cls"].str.startswith("G")
sample.loc[is_pool, "fold"] = sample.loc[is_pool, "appid"].map(fold_of_game).astype(int)
assert (sample["fold"] >= 0).all()
assert sample[is_pool].groupby("appid")["fold"].nunique().eq(1).all()


G1: objective 0.049
G2: objective 0.055
G3: objective 0.612
G4: objective 0.557
G5: objective 0.324


In [12]:
# fold audit: games, reviews and year coverage held out in each fold of each pool
rows = []
for (m, k), sub in sample[is_pool].groupby(["cls", "fold"]):
    yc = sub.groupby("year").size().reindex(YEARS, fill_value=0)
    rows.append(dict(cls=m, fold=k, games=sub["appid"].nunique(), reviews=len(sub),
                     years_covered=int((yc > 0).sum()), smallest_year=int(yc.min()),
                     held_out_games="; ".join(sorted(NAME2[a] for a in sub["appid"].unique())),
                     **{str(y): int(yc[y]) for y in YEARS}))
fold_audit = pd.DataFrame(rows)
fold_audit.to_csv(OUT / "fold_audit.csv", index=False)
fold_audit[["cls", "fold", "games", "reviews", "years_covered", "smallest_year", "held_out_games"]]


,cls,fold,games,reviews,years_covered,smallest_year,held_out_games
0,G1,0,8,20028,10,1713,Arma 3; BioShock Infinite; Dead Cells; Hunt: S...
1,G1,1,8,19914,10,1789,Counter-Strike: Source; Deceit; Deep Rock Gala...
2,G1,2,8,20137,10,1894,Black Mesa; Call of Duty®: Black Ops III; Dest...
3,G1,3,8,19830,10,1303,Counter-Strike; Cuphead; DOOM; Devil May Cry 5...
4,G1,4,8,20091,10,1869,Insurgency; Killing Floor 2; Mortal Kombat 11;...
5,G2,0,5,20303,10,1756,Life is Strange - Episode 1; NieR:Automata™; R...
6,G2,1,5,20371,10,1657,Assassin's Creed® Odyssey; Divinity: Original ...
7,G2,2,5,19922,10,1658,Just Cause™ 3; Kingdom Come: Deliverance; META...
8,G2,3,5,19482,10,1658,DARK SOULS™ III; Grand Theft Auto IV: The Comp...
9,G2,4,5,19922,10,1658,Assassin's Creed® Origins; Batman™: Arkham Kni...


In [13]:
# focal folds: size and year balance
focal_fold = sample[~is_pool].groupby(["cls", "fold"]).size().unstack("fold").reindex(CLASSES[:10])
print(focal_fold.to_string())
# training counts per class in each fold, the reason for equal class weighting
train_counts = pd.DataFrame({k: sample[sample["fold"] != k].groupby("cls").size() for k in range(N_FOLDS)}).reindex(CLASSES)
train_counts.columns = [f"train_fold{k}" for k in range(N_FOLDS)]
train_counts.to_csv(OUT / "train_counts_by_fold.csv")
train_counts


fold      0      1      2      3      4
cls                                    
F1    20001  20000  19998  20001  20000
F2    20001  19997  20000  20000  20002
F3    20000  19999  20001  20000  20000
F4    20000  20000  19999  19999  20002
F5    19999  19999  19999  20002  20001
F6    20002  19998  20000  20000  20000
F7    20003  20002  19999  19997  19999
F8    19999  20001  20000  20000  20000
F9    20001  20000  19998  20000  20001
F10   20000  19999  19999  20001  20001


,train_fold0,train_fold1,train_fold2,train_fold3,train_fold4
cls,,,,,
F1,79999,80000,80002,79999,80000
F2,79999,80003,80000,80000,79998
F3,80000,80001,79999,80000,80000
F4,80000,80000,80001,80001,79998
F5,80001,80001,80001,79998,79999
F6,79998,80002,80000,80000,80000
F7,79997,79998,80001,80003,80001
F8,80001,79999,80000,80000,80000
F9,79999,80000,80002,80000,79999


## 5. Texts and the saved sample

The review text is read a second time, only for the sampled rows, and saved with the metadata. The
column `emb_row` gives, for focal reviews, the row of `data/embeddings_v2.npz` holding that review's
vector; it is $-1$ for pool reviews, which are encoded on the cluster. `long_review` flags reviews
beyond the encoder's token window, so truncation can be reported by class and year.

In [14]:
sample = sample.sort_values(["cls", "appid", "src_pos"], key=lambda s: s.map(CLASS_IDX) if s.name == "cls" else s).reset_index(drop=True)
texts = np.empty(len(sample), dtype=object)
t0 = time.time()
for g in games.itertuples():
    pos = sample.index[sample["appid"] == g.appid]
    col = pd.read_feather(g.path, columns=["review"])["review"]
    texts[pos] = col.iloc[sample.loc[pos, "src_pos"].to_numpy()].fillna("").astype(str).str.strip().to_numpy()
    del col
sample["text"] = texts
del texts
gc.collect()
assert (sample["text"].str.count(r"\S+") == sample["word_count"]).all()
print(f"texts attached in {time.time() - t0:.0f}s")

sample["game"] = sample["appid"].map(NAME2)
sample["class_idx"] = sample["cls"].map(CLASS_IDX).astype(np.int8)
sample["long_review"] = sample["word_count"] > WORD_LIMIT
sample = sample[["cls", "class_idx", "appid", "game", "recommendationid", "year", "fold", "emb_row",
                 "word_count", "long_review", "edited", "voted_up", "votes_up", "votes_funny",
                 "weighted_vote_score", "text"]]
sample.to_parquet(OUT / "sample_15class.parquet", index=False)
print(f"saved {len(sample):,} reviews, {(OUT / 'sample_15class.parquet').stat().st_size / 1e6:.0f} MB")


texts attached in 9s
saved 1,500,000 reviews, 136 MB


In [15]:
realised = sample.pivot_table(index="cls", columns="year", values="appid", aggfunc="size").reindex(CLASSES).fillna(0).astype(int)
realised["total"] = realised.sum(1)
realised["games"] = sample.groupby("cls")["appid"].nunique().reindex(CLASSES)
realised["edited_share"] = sample.groupby("cls")["edited"].mean().reindex(CLASSES).round(3)
realised["long_share"] = sample.groupby("cls")["long_review"].mean().reindex(CLASSES).round(4)
realised.to_csv(OUT / "realised_class_year.csv")
assert realised["total"].eq(N_PER_CLASS).all() and len(sample) == 15 * N_PER_CLASS
realised


year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,total,games,edited_share,long_share
cls,,,,,,,,,,,,,,
F1,2821,3786,5142,5576,13638,23336,21777,18953,4971,0,100000,1,0.088,0.0066
F2,800,1342,1555,1425,4766,10934,17323,18173,22118,21564,100000,1,0.075,0.0009
F3,0,5786,4922,4951,10070,18161,22645,12863,11190,9412,100000,1,0.199,0.0075
F4,8743,8473,9707,4412,7191,12960,11378,10000,8977,18159,100000,1,0.114,0.0023
F5,14726,9165,12434,4763,7292,12394,15318,10927,9270,3711,100000,1,0.142,0.0025
F6,0,4173,22063,11043,13197,18909,12817,8586,5426,3786,100000,1,0.111,0.0025
F7,0,6735,4251,3862,11272,14374,16603,13746,13353,15804,100000,1,0.077,0.0014
F8,6346,14472,11700,5222,9393,16934,12999,9687,8230,5017,100000,1,0.083,0.0012
F9,0,0,837,6248,5950,14353,18325,31163,11816,11308,100000,1,0.174,0.0021


**What this shows.** The 15 classes each contribute exactly 100,000 distinct reviews, 1.5 million in
all. The five pools share one year distribution; the focal classes keep their own. The share of reviews
beyond the encoder's window is small in every class and is recorded here rather than assumed away.
The files in `results/design15/` are the inputs for the cluster step: `sample_15class.parquet` carries
the texts, folds and embedding rows; the CSVs are the audit tables for the paper's supplement.